# Simulated time series data

In [9]:
import numpy as np
import networkx as nx   

from sklearn.linear_model import LinearRegression
import pandas as pd

from tigramite import data_processing as pp
from tigramite import plotting as tp
from tigramite.rpcmci import RPCMCI
from tigramite.data_processing import DataFrame
from tigramite.independence_tests.parcorr import ParCorr
import pickle

from regime_selector import RegimeAICSelector

#### To strengthen the statistical validity of the simulation study, the authors should conduct Monte Carlo experiments with multiple independent replications for each synthetic system. Performance metrics such as regime recovery accuracy, classification accuracy, and forecasting error should be reported as averages across replications, accompanied by measures of variability (e.g., standard deviations or confidence intervals). This would allow readers to assess the stability and generalizability of the proposed method beyond single realizations.

# Monte Carlo Simulation Experiments

In [6]:
def structure_to_dag(structure):
    G = nx.DiGraph()

    for target, parents in structure.items():
        G.add_node(target)
        for parent_idx, lag in parents:
            parent = f"X{parent_idx}"
            G.add_edge(parent, target)

    return G

def percent_matching_edges(G_true, G_est):
    true_edges = set(G_true.edges())
    est_edges = set(G_est.edges())

    matches = true_edges & est_edges
    return len(matches) / len(true_edges) if len(true_edges) > 0 else 0.0

def get_causal_structure(n_regimes, results, rpcmci) -> dict:
    
    causal_structures = {}

    for w in range(n_regimes):
        graph_w = results['causal_results'][w]['graph'] # type: ignore
        val_w = results['causal_results'][w]['val_matrix'] # type: ignore

        parents_dict = rpcmci.return_parents_dict(
            graph=graph_w,
            val_matrix=val_w,
            include_lagzero_parents=False
        )

        causal_structures[w] = {
            f"X{int(target)}": [(int(i), int(tau)) for (i, tau) in parents]
            for target, parents in parents_dict.items()
        }

    return causal_structures


def causal_discovery(grid_search, dataframe):

    # -- RPCMCI for causal discovery with optimal Nk and Nc
    num_regimes = grid_search['best']['NK']
    max_transitions = grid_search['best']['NC']

    #print(f"Optimal Nk:{num_regimes}, Nc:{max_transitions}")

    rpcmci = RPCMCI(dataframe=dataframe, 
                cond_ind_test=ParCorr(),
                prediction_model=LinearRegression(),
                verbosity= -2)

    # RPCMCI parameters
    switch_thres = 0.05
    num_iterations = 20
    max_anneal = 10
    tau_min = 1
    tau_max = 1
    pc_alpha = 0.1
    alpha_level = 0.01
    n_jobs = -1       


    # Run RPCMCI
    results = rpcmci.run_rpcmci(
        num_regimes=num_regimes, 
        max_transitions=max_transitions, 
        switch_thres=switch_thres, 
        num_iterations=num_iterations, 
        max_anneal=max_anneal, 
        tau_min=tau_min, 
        tau_max=tau_max,
        pc_alpha=pc_alpha, 
        alpha_level=alpha_level, 
        n_jobs=n_jobs
        )
    
    # did not converge
    if results is None:
        print("no convergence")
        return None, None, None, None
    
    regimes = results['regimes'].argmax(axis=0)  # type: ignore
    n_regimes = len(results['causal_results']) # type: ignore

    if regimes[0] == 1:
        regimes = 1 - regimes
        results["causal_results"] = [
            results["causal_results"][1],
            results["causal_results"][0],
        ]
    
    return regimes, n_regimes, rpcmci, results

In [7]:
def simulation(seed=1):
    print(f"---- Monte Carlo Simulation {seed} ----")

    # Set seed
    np.random.seed(seed)

    # Time settings
    T = 700
    num_vars = 3
    var_names = ['X0', 'X1', 'X2']
    # Initialize data
    data = np.random.randn(T, num_vars)
    true_regimes = np.zeros(T, dtype=int)

    # Define regime periods
    regime_0_intervals = [(0, 200), (606, 699)]
    regime_1_intervals = [(201, 605)]

    # Mark regimes
    for start, end in regime_0_intervals:
        true_regimes[start:end + 1] = 0
    for start, end in regime_1_intervals:
        true_regimes[start:end + 1] = 1

    # Simulate data based on structural equations
    for t in range(1, T):
        if true_regimes[t] == 0:
            # Regime 0
            data[t, 0] += 0.4 * data[t - 1, 0] + 0.3 * data[t - 1, 1]
            data[t, 1] += 0.3 * data[t - 1, 1] + 0.6 * data[t - 1, 2]
            data[t, 2] += 0.3 * data[t - 1, 2] + 0.7 * data[t - 1, 0]
        else:
            # Regime 1
            data[t, 0] += 0.4 * data[t - 1, 0] - 0.5 * data[t - 1, 2]
            data[t, 1] += 0.3 * data[t - 1, 1]
            data[t, 2] += 0.3 * data[t - 1, 2] - 0.8 * data[t - 1, 1]

    # Convert to DataFrame
    df = pd.DataFrame(data, columns=var_names)

    # Training data
    train_data = data[:500]

    # Grid search for optimal Nk and Nc
    selector = RegimeAICSelector(
        data=train_data,
        tau_max=1,
        switch_thres=0.05,
        num_iterations=20,
        max_anneal=10,
        pc_alpha=0.2,
        alpha_level=0.01,
        n_jobs=-1,
        seed=1,
        cond_ind_test=ParCorr(),
        prediction_model=LinearRegression(),
        verbosity=-2
    )

    grid_search = selector.find_best(grid_NK=range(2, 4), grid_NC=range(1, 4))

    # RPCMCI for causal discovery
    dataframe = pp.DataFrame(train_data)
    regimes, n_regimes, rpcmci, results = causal_discovery(grid_search, dataframe)

    if n_regimes is None:
        return "Did not converge"

    causal_structure = get_causal_structure(n_regimes, results, rpcmci)

    # Use estimated regimes for train, true regimes for test
    df["regime"] = np.concatenate((regimes, true_regimes[500:])) # type: ignore

    target_structure = {
        0: {
    'X0': [(0, -1), (1, -1)], 
    'X1': [(2, -1), (1, -1)], 
    'X2': [(0, -1), (2, -1)]}, 
       1: {
     'X0': [(2, -1), (0, -1)],
     'X1': [(1, -1)],
     'X2': [(1, -1), (2, -1)]}}
    
    target0 = structure_to_dag(target_structure[0])
    target1 = structure_to_dag(target_structure[1])

    estimated0 = structure_to_dag(causal_structure[0])
    estimated1 = structure_to_dag(causal_structure[1])

    pct0 = percent_matching_edges(target0, estimated0)
    pct1 = percent_matching_edges(target1, estimated1)
    overall_pct = (pct0 + pct1) / 2

   

    best_nc = grid_search["best"]["NC"]
    best_nk = grid_search["best"]["NK"]
    nk_is_2 = best_nk == 2
    nc_is_1 = best_nc == 1

    print(f"Nk == 2: {nk_is_2}")
    print(f"Nc == 1: {nc_is_1}")
    print(f"Estimated causal graph accuracy: {overall_pct}")

    return {
        "seed": seed,
        "df": df,
        "causal_structure": causal_structure,
        "nk_is_2": nk_is_2,
        "nc_is_3": nc_is_1,
        "structure_match": overall_pct}

In [8]:
seeds =  [3, 11, 17, 24, 75]
causal_discovery_results = [simulation(seed) for seed in seeds]

---- Monte Carlo Simulation 3 ----
Nk == 2: True
Nc == 1: True
Estimated causal graph accuracy: 1.0
---- Monte Carlo Simulation 11 ----
No annealings have converged. Run failed.
No annealings have converged. Run failed.
Nk == 2: True
Nc == 1: True
Estimated causal graph accuracy: 1.0
---- Monte Carlo Simulation 17 ----
No annealings have converged. Run failed.
No annealings have converged. Run failed.
Nk == 2: True
Nc == 1: True
Estimated causal graph accuracy: 1.0
---- Monte Carlo Simulation 24 ----
No annealings have converged. Run failed.
No annealings have converged. Run failed.
No annealings have converged. Run failed.
Nk == 2: True
Nc == 1: True
Estimated causal graph accuracy: 1.0
---- Monte Carlo Simulation 75 ----
No annealings have converged. Run failed.
No annealings have converged. Run failed.
No annealings have converged. Run failed.
Nk == 2: True
Nc == 1: True
Estimated causal graph accuracy: 1.0


In [11]:
# save regime classification and causal structure
with open("mc_system3.pkl", "wb") as f:
    pickle.dump(causal_discovery_results, f)